# 02_ex_<agreement>_<topic> — exploration and approval drafting

Use this notebook to profile source data, draft AI-assisted suggestions, and document human-reviewed decisions before anything is enforced in pipeline execution.

**You edit**
- Agreement/topic metadata
- Source target details
- Draft metadata handover, DQ, and classification decisions

**This notebook produces**
- Profiling evidence
- Candidate DQ/classification suggestions
- Human-reviewed metadata handover draft and approved decisions

**AI advisory boundary**
- AI suggestions are advisory and must be human-reviewed before promotion.


## Segment 1: Load shared config and imports

What this does: loads `00_env_config` and imports exploration helpers.

Functions used: `setup_fabricops_notebook`, `load_fabric_config`, `generate_metadata_profile`, `draft_dq_rules`.

Output: configured runtime context and helper availability.


In [ ]:
%run 00_env_config


In [ ]:
import json
from pathlib import Path
import fabricops_kit.data_quality as data_quality
from fabricops_kit.config import setup_fabricops_notebook, get_path, load_fabric_config
from fabricops_kit.fabric_input_output import seed_minimal_sample_source_table
from fabricops_kit.fabric_input_output import lakehouse_table_read, warehouse_read
from fabricops_kit.data_profiling import profile_dataframe_to_metadata
from fabricops_kit import (
    draft_dq_rules,
    review_dq_rules,
    write_dq_rules,
)


## Agreement and approved usage


In [ ]:
USE_SAMPLE_DATA = True
DATA_AGREEMENT = "sample_minimal_agreement" if USE_SAMPLE_DATA else "TODO_replace_agreement"
APPROVED_USAGE = "template_proof_path" if USE_SAMPLE_DATA else "TODO_replace_approved_usage"
ENV_NAME = "dev"
SOURCE_LAYER = "source"
SOURCE_KIND = "lakehouse"
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "topic_dataset"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else f"{DATASET_NAME}_output"
DQ_TABLE_NAME = TARGET_TABLE


In [ ]:
CONFIG = load_fabric_config(CONFIG)
if not USE_SAMPLE_DATA:
    setup_fabricops_notebook(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])


In [ ]:
source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG) if not USE_SAMPLE_DATA else None


In [ ]:
if SOURCE_KIND == "lakehouse":
    if USE_SAMPLE_DATA:
        source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
        seed_minimal_sample_source_table(source_path, table_name=SOURCE_TABLE)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
print(df_source.schema)
display(df_source.limit(20))


## Segment 2: Profile source and capture evidence

What this does: loads source data and builds profiling metadata for review.

Functions used: `lakehouse_table_read`/`warehouse_read`, `generate_metadata_profile`, `profile_dataframe_to_metadata`.

Output: source profile rows for human review and downstream drafting.


In [ ]:
profile_rows = profile_dataframe_to_metadata(df_source, table_name=DQ_TABLE_NAME)
display(profile_rows)


## Exploration findings
Document EDA findings here.


## Transformation rationale
Document approved transformation rationale for pipeline handoff.


## Segment 3: AI-assisted DQ drafting (advisory only)

What this does: uses profile metadata plus configured prompts to generate advisory candidate DQ rules.

Functions used: `draft_dq_rules`, `review_dq_rules`, `write_dq_rules`.


In [ ]:
CANDIDATE_DQ_RULES = draft_dq_rules(
    profile_df=profile_rows,
    table_name=DQ_TABLE_NAME,
    business_context="TODO_replace_business_context",
    prompt_template=CONFIG.ai_prompt_config.dq_rule_candidate_template,
    output_col="ai_response",
)
review_dq_rules(CANDIDATE_DQ_RULES, table_name=DQ_TABLE_NAME)


## Human-approved DQ rules for pipeline enforcement
Keep only approved deterministic rules for DQ metadata writes used by 03_pc.


In [ ]:
# Review widget writes explicit approvals to data_quality.APPROVED_RULES_FROM_WIDGET.
HUMAN_APPROVED_RULES = list(data_quality.APPROVED_RULES_FROM_WIDGET)
if not HUMAN_APPROVED_RULES:
    raise ValueError("No approved DQ rules selected in widget. Approve at least one rule before persisting.")

metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)
approved_rules_metadata_df = write_dq_rules(
    HUMAN_APPROVED_RULES,
    table_name=DQ_TABLE_NAME,
    metadata_path=metadata_path,
    action_by="notebook_user",
)
display(approved_rules_metadata_df)


## Human-reviewed DQ decisions
Copy only approved deterministic rules to 03_pc notebook.


In [ ]:
# Optional: use this section when this workflow is needed.
# Governance/classification example helpers:
# manual_pkg = build_manual_governance_prompt_package(profile={"columns": list(df_source.columns)})
# ai_candidates = generate_governance_candidates_with_fabric_ai(profile={"columns": list(df_source.columns)}, config=CONFIG)
# parsed = parse_manual_ai_json_response("{}")
# classifications = classify_columns(list(df_source.columns))
# records = build_governance_classification_records(DATASET_NAME, SOURCE_TABLE, classifications, run_id=RUN_ID)
# write_governance_classifications(records, metadata_path=get_path(ENV_NAME, "metadata", config=CONFIG))
# summarize_governance_classifications(classifications)
# Handover narrative example helpers:
# handover_pkg = build_manual_handover_prompt_package(context={"dataset": DATASET_NAME, "table": SOURCE_TABLE})
# handover_summary = generate_handover_summary_with_fabric_ai(context={"dataset": DATASET_NAME, "table": SOURCE_TABLE}, config=CONFIG)


## Segment 4: Human approval and DQ metadata write

What this does: writes human-approved deterministic DQ rules to metadata for pipeline enforcement.

Functions used: `write_dq_rules`.

Output: approved DQ metadata used by 03_pc pipeline enforcement.


## Human-reviewed classification decisions
Approve labels with governance stewards.


In [ ]:
# Optional governance evidence omitted for minimal path.


## Persist approved DQ metadata
Writes approved DQ rules; governance/classification/drift metadata are future TODO workflows.


In [ ]:
USE_LOCAL_SAMPLE_METADATA = False
# Set True only when running locally without Fabric metadata tables.
if not USE_LOCAL_SAMPLE_METADATA:
    metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)
    # TODO: persist approved profiling, DQ, governance, and drift metadata via dedicated helpers.
    # Handover joins those approved metadata tables into the final table agreement view.


In [ ]:
from fabricops_kit.data_lineage import build_lineage_from_notebook_code

# Optional documentation helper only (not transformation logic).
# Paste or load exported notebook source when ready to document lineage.
# result = build_lineage_from_notebook_code(notebook_code)
# display(result["deterministic_steps"])
# print(result["copilot_prompt"])
